# Mini-GPT: Alpaca Q&A Fine-Tuning

This notebook fine-tunes the pretrained `run_02` checkpoint on the **Alpaca instruction-following dataset** so the model answers questions instead of just completing text.

### What changes vs. pretraining
| | Pretraining (run_02) | Fine-tuning (this notebook) |
|---|---|---|
| Data | OpenWebText (raw text) | Alpaca 52K instruction/response pairs |
| Objective | Predict every next token | Predict **response tokens only** (instruction masked) |
| LR | 3e-4 | 5e-5 (10× lower — protects pretrained weights) |
| Epochs | 3 | 3–5 |

### Output
Saves a new export folder `run_03_alpaca/` with:
- `mini_gpt_state.pt` — fine-tuned weights  
- `mini_gpt_config.json` — same architecture config  
- `tokenizer/tokenizer.json` — same tokenizer  

This drop-in replaces `run_02` in the Streamlit app — just select `run_03_alpaca` in the sidebar.

## 1. Setup

In [ ]:
import importlib, subprocess, sys

required = {"datasets": "datasets", "tokenizers": "tokenizers", "tqdm": "tqdm"}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

import json, math, os, platform, random, time
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tokenizers import Tokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load run_02 Artifacts

Upload the `run_02` folder (containing `mini_gpt_state.pt`, `mini_gpt_config.json`, `tokenizer/tokenizer.json`) to Colab, or mount Google Drive.

In [ ]:
# ── Option A: Mount Google Drive ──────────────────────────────────────────
# from google.colab import drive
# drive.mount("/content/drive")
# RUN_02_DIR = Path("/content/drive/MyDrive/Mini-GPT/run_02")  # adjust path

# ── Option B: Upload directly (uploads zip then unzips) ───────────────────
# from google.colab import files
# uploaded = files.upload()   # upload run_02.zip
# import zipfile
# with zipfile.ZipFile("run_02.zip") as z: z.extractall("/content/run_02")
# RUN_02_DIR = Path("/content/run_02")

# ── Option C: Already on disk (local / Kaggle) ────────────────────────────
RUN_02_DIR = Path("/content/run_02")   # ← change if needed

assert (RUN_02_DIR / "mini_gpt_state.pt").exists(), \
    f"Not found: {RUN_02_DIR / 'mini_gpt_state.pt'} — check RUN_02_DIR"

print(f"run_02 dir: {RUN_02_DIR}")
print("Files:", [f.name for f in RUN_02_DIR.rglob("*") if f.is_file()])

## 3. Mini-GPT Architecture (copy from repo)

In [ ]:
# ── Self-contained model definition (mirrors models/ in the repo) ──────────

@dataclass
class GPTConfig:
    vocab_size: int
    embedding_dim: int
    num_heads: int
    num_layers: int
    context_length: int
    dropout: float = 0.1


class TokenPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_length, dropout):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        return self.dropout(self.token_embedding(input_ids) + self.position_embedding(positions))


class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout):
        super().__init__()
        assert embedding_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(context_length, context_length)).unsqueeze(0).unsqueeze(0)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).split(C, dim=-1)
        def reshape(t): return t.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        q, k, v = map(reshape, qkv)
        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, embedding_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.GELU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.attn = MultiHeadCausalSelfAttention(embedding_dim, num_heads, context_length, dropout)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.ff = FeedForward(embedding_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.embedding = TokenPositionEmbedding(
            config.vocab_size, config.embedding_dim, config.context_length, config.dropout
        )
        self.blocks = nn.ModuleList([
            TransformerBlock(config.embedding_dim, config.num_heads, config.context_length, config.dropout)
            for _ in range(config.num_layers)
        ])
        self.final_ln = nn.LayerNorm(config.embedding_dim)
        self.lm_head = nn.Linear(config.embedding_dim, config.vocab_size)

    def forward(self, input_ids, targets=None):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.final_ln(x))
        loss = None
        if targets is not None:
            # IGNORE_INDEX=-100 masks the instruction tokens from loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss


print("Model classes defined.")

## 4. Load Pretrained Checkpoint

In [ ]:
# ── Load config ────────────────────────────────────────────────────────────
with open(RUN_02_DIR / "mini_gpt_config.json") as f:
    cfg_dict = json.load(f)

config = GPTConfig(
    vocab_size=cfg_dict["vocab_size"],
    embedding_dim=cfg_dict["embedding_dim"],
    num_heads=cfg_dict["num_heads"],
    num_layers=cfg_dict["num_layers"],
    context_length=cfg_dict["context_length"],
    dropout=cfg_dict.get("dropout", 0.1),
)
print("Config:", config)

# ── Build model ────────────────────────────────────────────────────────────
model = MiniGPT(config).to(DEVICE)

# ── Load weights (remap notebook key names → repo key names) ───────────────
state_dict = torch.load(RUN_02_DIR / "mini_gpt_state.pt", map_location=DEVICE)
key_map = {
    "embedding.token_emb.weight": "embedding.token_embedding.weight",
    "embedding.pos_emb.weight": "embedding.position_embedding.weight",
}
state_dict = {key_map.get(k, k): v for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Loaded checkpoint. Missing keys: {missing}  |  Unexpected: {unexpected}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# ── Load tokenizer ─────────────────────────────────────────────────────────
tok_path = RUN_02_DIR / "tokenizer" / "tokenizer.json"
tokenizer = Tokenizer.from_file(str(tok_path))
tokenizer.enable_truncation(max_length=config.context_length)
print(f"Tokenizer vocab size: {tokenizer.get_vocab_size()}")

## 5. Alpaca Dataset — Load and Format

The Alpaca dataset has 52K instruction/input/output triples. We format each as:
```
### Instruction:
{instruction}\n\n### Input:\n{input}\n\n### Response:\n{output}
```
Loss is computed **only on the `### Response:` part** — the instruction tokens are masked with -100.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MAX_SAMPLES   = 52000   # use all 52K  (set lower, e.g. 10000, for a quick test)
PROMPT_SUFFIX = "### Response:\n"   # everything AFTER this is response tokens

def build_prompt(instruction: str, input_text: str, output: str) -> tuple[str, str]:
    """Returns (full_text, prompt_prefix_only)."""
    if input_text.strip():
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n"
    prompt += PROMPT_SUFFIX
    full_text = prompt + output
    return full_text, prompt


print("Loading tatsu-lab/alpaca …")
raw_dataset = load_dataset("tatsu-lab/alpaca", split="train")
print(f"Total Alpaca samples: {len(raw_dataset)}")

# Show a sample
ex = raw_dataset[0]
full, prefix = build_prompt(ex["instruction"], ex.get("input", ""), ex["output"])
print("\n--- Sample prompt ---")
print(full[:400])

## 6. Build Token Dataset with Loss Masking

In [ ]:
IGNORE_INDEX = -100
CTX = config.context_length   # 256 tokens


class AlpacaDataset(TorchDataset):
    """
    Each item: (input_ids [CTX], labels [CTX])
    Labels for instruction tokens are set to IGNORE_INDEX so the model
    only learns from the response part.
    """

    def __init__(self, raw_data, tokenizer: Tokenizer, max_samples: int = None):
        self.samples: list[tuple[list[int], list[int]]] = []
        skipped = 0
        limit = min(max_samples or len(raw_data), len(raw_data))

        for row in tqdm(raw_data.select(range(limit)), desc="Tokenising"):
            full_text, prompt_prefix = build_prompt(
                row["instruction"], row.get("input", ""), row["output"]
            )

            full_ids   = tokenizer.encode(full_text).ids
            prefix_ids = tokenizer.encode(prompt_prefix).ids

            # Skip samples where full text is longer than context window
            # (we could truncate, but that may cut off the response)
            if len(full_ids) < 4:   # too short
                skipped += 1
                continue

            # Truncate to context length
            full_ids = full_ids[:CTX]
            prefix_len = min(len(prefix_ids), len(full_ids))

            # Pad to CTX
            pad_len = CTX - len(full_ids)
            input_ids = full_ids + [0] * pad_len

            # Labels: mask instruction tokens and padding
            labels = [IGNORE_INDEX] * prefix_len + full_ids[prefix_len:] + [IGNORE_INDEX] * pad_len

            # Shift: input = tokens[:-1], labels = tokens[1:]
            input_ids_shifted = input_ids[:-1]
            labels_shifted = labels[1:]

            self.samples.append((input_ids_shifted, labels_shifted))

        print(f"Dataset built: {len(self.samples)} samples  ({skipped} skipped)")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        input_ids, labels = self.samples[idx]
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels, dtype=torch.long),
        )


dataset = AlpacaDataset(raw_dataset, tokenizer, max_samples=MAX_SAMPLES)

# ── Train / val split (90 / 10) ───────────────────────────────────────────
val_size  = max(100, int(0.1 * len(dataset)))
train_size = len(dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}")

## 7. Fine-Tuning Config

In [ ]:
# ── Hyperparameters — feel free to tweak ──────────────────────────────────
BATCH_SIZE     = 16       # reduce to 8 if GPU OOM
LEARNING_RATE  = 5e-5     # 10× lower than pretraining
EPOCHS         = 3
WARMUP_STEPS   = 50
GRAD_CLIP      = 1.0
GRAD_ACCUM     = 2        # effective batch = BATCH_SIZE * GRAD_ACCUM
EVAL_INTERVAL  = 100      # eval every N optimizer steps
WEIGHT_DECAY   = 0.01

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
)

total_steps   = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
scaler        = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print(f"Total optimiser steps: {total_steps}")
print(f"Effective batch size:  {BATCH_SIZE * GRAD_ACCUM}")

## 8. Fine-Tuning Loop

In [ ]:
def evaluate(model, loader, max_batches=30):
    model.eval()
    total_loss, count = 0.0, 0
    with torch.no_grad():
        for i, (input_ids, labels) in enumerate(loader):
            if i >= max_batches: break
            input_ids = input_ids.to(DEVICE)
            labels    = labels.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                _, loss = model(input_ids, labels)
            if loss is not None:
                total_loss += loss.item()
                count += 1
    model.train()
    return total_loss / max(count, 1)


history = {"train_loss": [], "val_loss": [], "step": []}
global_step = 0
best_val_loss = float("inf")

model.train()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    optimizer.zero_grad()

    for step, (input_ids, labels) in enumerate(pbar, 1):
        input_ids = input_ids.to(DEVICE)
        labels    = labels.to(DEVICE)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            _, loss = model(input_ids, labels)

        loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % EVAL_INTERVAL == 0:
                val_loss = evaluate(model, val_loader)
                avg_train = epoch_loss / step
                history["train_loss"].append(avg_train)
                history["val_loss"].append(val_loss)
                history["step"].append(global_step)
                pbar.set_postfix({"train_loss": f"{avg_train:.4f}", "val_loss": f"{val_loss:.4f}"})

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), "/tmp/best_checkpoint.pt")

    print(f"Epoch {epoch} done. Avg loss: {epoch_loss/len(train_loader):.4f}")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

## 9. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

if history["step"]:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history["step"], history["train_loss"], label="Train Loss")
    ax.plot(history["step"], history["val_loss"],   label="Val Loss")
    ax.set_xlabel("Optimiser Step")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.set_title("Alpaca Fine-Tuning — Loss Curves")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No eval data yet — run more steps or reduce EVAL_INTERVAL.")

## 10. Save Fine-Tuned Checkpoint

Saves in the same format as `run_02` — drop-in compatible with the Streamlit app.

In [ ]:
import shutil

# ── Set output directory ───────────────────────────────────────────────────
# For Google Drive:
# OUT_DIR = Path("/content/drive/MyDrive/Mini-GPT/run_03_alpaca")
OUT_DIR = Path("/content/run_03_alpaca")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tokenizer").mkdir(exist_ok=True)

# ── Load best weights saved during training ───────────────────────────────
best_state = torch.load("/tmp/best_checkpoint.pt", map_location="cpu")
torch.save(best_state, OUT_DIR / "mini_gpt_state.pt")
print(f"Saved: {OUT_DIR / 'mini_gpt_state.pt'}")

# ── Copy model config ──────────────────────────────────────────────────────
shutil.copy(RUN_02_DIR / "mini_gpt_config.json", OUT_DIR / "mini_gpt_config.json")
print(f"Saved: {OUT_DIR / 'mini_gpt_config.json'}")

# ── Copy tokenizer ─────────────────────────────────────────────────────────
shutil.copy(RUN_02_DIR / "tokenizer" / "tokenizer.json", OUT_DIR / "tokenizer" / "tokenizer.json")
if (RUN_02_DIR / "tokenizer" / "tokenizer_config.json").exists():
    shutil.copy(RUN_02_DIR / "tokenizer" / "tokenizer_config.json", OUT_DIR / "tokenizer" / "tokenizer_config.json")
print(f"Saved: tokenizer")

# ── Save fine-tune metadata ────────────────────────────────────────────────
ft_meta = {
    "base_checkpoint": "run_02",
    "fine_tune_dataset": "tatsu-lab/alpaca",
    "fine_tune_samples": MAX_SAMPLES,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "best_val_loss": best_val_loss,
    "prompt_format": "### Instruction:\n{instruction}\n\n### Response:\n{output}",
}
with open(OUT_DIR / "finetune_config.json", "w") as f:
    json.dump(ft_meta, f, indent=2)
print(f"Saved: finetune_config.json")

# ── Zip for download ───────────────────────────────────────────────────────
zip_out = "/content/run_03_alpaca"
shutil.make_archive(zip_out, "zip", "/content", "run_03_alpaca")
print(f"\nDownload ready: {zip_out}.zip")

# Uncomment to download automatically in Colab:
# from google.colab import files
# files.download(zip_out + ".zip")

## 11. Quick Inference Test

Test that the model now answers questions in Q&A format.

In [ ]:
@torch.no_grad()
def answer_question(
    question: str,
    max_new_tokens: int = 150,
    temperature: float = 0.7,
    top_k: int = 40,
    top_p: float = 0.9,
    repetition_penalty: float = 1.2,
    context_input: str = "",
):
    """
    Format a question using the Alpaca prompt template and generate a response.
    Includes safety checks to avoid CUDA device-side asserts from invalid sampling states.
    """
    if context_input.strip():
        prompt = f"### Instruction:\n{question}\n\n### Input:\n{context_input}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{question}\n\n### Response:\n"

    model.eval()

    prompt_ids = tokenizer.encode(prompt).ids
    vocab_size = config.vocab_size
    invalid_ids = [tid for tid in prompt_ids if tid < 0 or tid >= vocab_size]
    if invalid_ids:
        raise ValueError(
            f"Tokenizer produced out-of-range token ids (max valid id: {vocab_size - 1}). "
            f"Sample invalid ids: {invalid_ids[:10]}"
        )

    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    generated = input_ids[0].tolist()

    for _ in range(max_new_tokens):
        ctx_ids = torch.tensor([generated[-config.context_length :]], dtype=torch.long, device=DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits, _ = model(ctx_ids)

        logits = logits[0, -1, :].float()

        # Repetition penalty
        if repetition_penalty != 1.0:
            for token_id in set(generated):
                if 0 <= token_id < logits.numel():
                    if logits[token_id] > 0:
                        logits[token_id] /= repetition_penalty
                    else:
                        logits[token_id] *= repetition_penalty

        logits = logits / max(temperature, 1e-6)

        # Top-k
        if top_k > 0:
            k = min(top_k, logits.size(-1))
            kth = torch.topk(logits, k).values[-1]
            logits = torch.where(logits < kth, torch.full_like(logits, float("-inf")), logits)

        # Top-p (nucleus) with stable mask-shift so at least one token is kept
        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
            sorted_indices_to_remove[0] = False

            sorted_logits = sorted_logits.masked_fill(sorted_indices_to_remove, float("-inf"))
            logits = torch.zeros_like(logits).scatter_(0, sorted_idx, sorted_logits)

        # Fallback protection: avoid all -inf / NaN distributions on GPU
        if not torch.isfinite(logits).any():
            logits = torch.zeros_like(logits)

        probs = F.softmax(logits, dim=-1)
        if (not torch.isfinite(probs).all()) or probs.sum() <= 0:
            next_token = torch.argmax(logits).item()
        else:
            next_token = torch.multinomial(probs, num_samples=1).item()

        generated.append(next_token)

    full_text = tokenizer.decode(generated)
    if "### Response:\n" in full_text:
        response = full_text.split("### Response:\n", 1)[1].strip()
    else:
        response = full_text[len(prompt) :].strip()
    return response


# Test questions
test_questions = [
    "What is machine learning?",
    "Explain how photosynthesis works.",
    "Write a short poem about the sea.",
    "What are the three laws of motion?",
]

for q in test_questions:
    print(f"Q: {q}")
    response = answer_question(q)
    print(f"A: {response}")
    print("-" * 60)

## 12. Use Fine-Tuned Model in the Streamlit App

After downloading `run_03_alpaca.zip`:

1. **Unzip** it into your `d:\Mini-GPT\` folder so the path is:
   ```
   d:\Mini-GPT\run_03_alpaca\mini_gpt_state.pt
   d:\Mini-GPT\run_03_alpaca\mini_gpt_config.json
   d:\Mini-GPT\run_03_alpaca\tokenizer\tokenizer.json
   ```

2. **Run Streamlit** — the app auto-discovers export folders:
   ```bash
   streamlit run streamlit_app.py
   ```

3. **Select `run_03_alpaca`** in the sidebar model dropdown.

4. **Change your prompt style** — use the Alpaca format for best results:
   ```
   ### Instruction:
   What is deep learning?
   
   ### Response:
   ```

5. **Recommended hyperparameters** for Q&A (lower temperature for factual answers):
   - Temperature: **0.7**
   - Top-k: **40**
   - Top-p: **0.90**
   - Repetition penalty: **1.2**